In [ ]:
import importlib.util
import re
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
DIET_CSV = DATA_DIR / "csv_diet" / "select_diet_oral_baseline_participants_diet_logging.csv"
REPO_DIR = Path.cwd() if (Path.cwd() / "dietary_score_definitions").is_dir() else Path.cwd().parent
CONFIG = REPO_DIR / "dietary_score_definitions" / "diet_score_config.py"
SUPP2_CSV = REPO_DIR / "dietary_score_definitions" / "supplementary2.csv"
OUT_DIR = DATA_DIR / "diet_experiment"

NUTRIENTS = {
    "carb_g": "carbohydrate_g", "fat_g": "lipid_g", "protein_g": "protein_g",
    "fiber_g": "dietary_fiber_g", "sodium_mg": "sodium_mg", "alcohol_g": "alcohol_g",
}
FOOD, CATEGORY, WEIGHT, ENERGY = "short_food_name", "food_category", "weight_g", "calories_kcal"

spec = importlib.util.spec_from_file_location("diet_score_config", CONFIG)
cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)


def canon(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())


def safe_name(x):
    return re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z_]+", "_", str(x).strip())).strip("_")[:120]


def safe_divide(a, b):
    return (a / b.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)


def cv(x):
    m = x.mean()
    return 0 if pd.isna(m) or m == 0 else x.std() / m


def std0(x):
    v = x.std()
    return 0 if pd.isna(v) else v


def linear_positive(x, low, high, max_score=10):
    return ((x.fillna(0) - low) / (high - low) * max_score).clip(0, max_score)


def linear_negative(x, low, high, max_score=10):
    return ((high - x.fillna(0)) / (high - low) * max_score).clip(0, max_score)


def cols_sum(df, names):
    wanted = {canon(n) for n in names}
    cols = [c for c in df.columns if canon(c) in wanted]
    return df[cols].sum(axis=1) if cols else pd.Series(0, index=df.index)


def quintile_positive(s):
    s = s.fillna(0)
    if s.nunique() < 2:
        return pd.Series(1, index=s.index, dtype=float)
    # Score intake values directly so ties never depend on participant order.
    cutpoints = s.quantile([0.2, 0.4, 0.6, 0.8]).to_numpy()
    # Boundary ties stay in the lower quintile; repeated cutpoints can leave bins empty.
    scores = 1 + np.searchsorted(cutpoints, s.to_numpy(), side="left")
    return pd.Series(scores, index=s.index, dtype=float)


def eat_component_score(x, rule):
    score = pd.Series(0, index=x.index, dtype=float)
    for points in [3, 2, 1, 0]:
        low, high = rule[f"score_{points}"]
        mask = x < high if low is None else (x >= low if high is None else (x >= low) & (x < high))
        score[mask] = points
    return score


diet = pd.read_csv(DIET_CSV, low_memory=False)
supp2 = pd.read_csv(SUPP2_CSV, index_col=0)
supp2.index = supp2.index.astype(str).str.strip()
supp2.columns = supp2.columns.astype(str).str.strip()

diet["participant_id"] = diet["participant_id"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
diet["_date"] = pd.to_datetime(diet["collection_date"], errors="coerce").dt.date
diet["_timestamp"] = pd.to_datetime(diet["collection_timestamp"], errors="coerce", utc=True)
for c in [WEIGHT, ENERGY, *NUTRIENTS.values()]:
    diet[c] = pd.to_numeric(diet[c], errors="coerce").fillna(0)

In [ ]:
day_keys = ["participant_id", "_date"]
daily = diet.groupby(day_keys)[ENERGY].sum().rename("daily_energy_kcal").reset_index()
daily = daily.merge(diet.groupby(day_keys).agg(
    diet_event_count=(FOOD, "size"), foods_per_day=(FOOD, "nunique"), categories_per_day=(CATEGORY, "nunique"),
).reset_index(), on=day_keys, how="left")

timed = diet.dropna(subset=["_timestamp"]).assign(_hour=lambda d: d["_timestamp"].dt.floor("h"))
timing = timed.groupby(day_keys).agg(
    meals_per_day=("_hour", "nunique"), first=("_timestamp", "min"), last=("_timestamp", "max")).reset_index()
timing["eating_window_hours"] = (timing["last"] - timing["first"]).dt.total_seconds() / 3600
timing["fasting_window_hours"] = 24 - timing["eating_window_hours"]
daily = daily.merge(timing.drop(columns=["first", "last"]), on=day_keys, how="left")
daily["foods_per_meal"] = safe_divide(daily["foods_per_day"], daily["meals_per_day"])

behavior = daily.groupby("participant_id").agg(
    diet_logging_days=("_date", "nunique"),
    Energy=("daily_energy_kcal", "mean"), Energy_std=("daily_energy_kcal", std0), Energy_cv=("daily_energy_kcal", cv),
    diet_event_count_mean=("diet_event_count", "mean"), diet_event_count_std=("diet_event_count", std0),
    Foods_per_day=("foods_per_day", "mean"), Foods_per_day_std=("foods_per_day", std0), Foods_per_day_cv=("foods_per_day", cv),
    Categories_per_day=("categories_per_day", "mean"), Categories_per_day_std=("categories_per_day", std0),
    Meals_per_day=("meals_per_day", "mean"), Meals_per_day_std=("meals_per_day", std0),
    Foods_per_meal=("foods_per_meal", "mean"), Foods_per_meal_std=("foods_per_meal", std0),
    Fasting_window=("fasting_window_hours", "mean"), Fasting_window_std=("fasting_window_hours", std0),
    Eating_window_hours=("eating_window_hours", "mean"),
).reset_index()

In [ ]:
person_days = diet.groupby("participant_id")["_date"].nunique()
person_energy = diet.groupby("participant_id")[ENERGY].sum()


def energy_features(frame, column, prefix):
    total = frame.pivot_table(index="participant_id", columns=column, values=ENERGY, aggfunc="sum", fill_value=0)
    daily_kcal = total.div(person_days.reindex(total.index), axis=0)
    energy_pct = 100 * total.div(person_energy.reindex(total.index), axis=0).fillna(0)
    daily_kcal.columns = [f"{prefix}_daily_kcal_{safe_name(c)}" for c in total.columns]
    energy_pct.columns = [f"{prefix}_energy_pct_{safe_name(c)}" for c in total.columns]
    return pd.concat([daily_kcal, energy_pct], axis=1).reset_index(), total


food_presence = diet[["participant_id", FOOD]].dropna().drop_duplicates().groupby(FOOD)["participant_id"].nunique()
common_foods = food_presence[food_presence >= max(1, int(np.ceil(0.01 * diet["participant_id"].nunique())))].index
food_features, _ = energy_features(diet[diet[FOOD].isin(common_foods)], FOOD, "food")
category_features, category_energy_total = energy_features(diet, CATEGORY, "category")

nutrient_cols = [ENERGY, WEIGHT, *NUTRIENTS.values()]
daily_nutr = diet.groupby(day_keys)[nutrient_cols].sum().reset_index()
nutr_mean = daily_nutr.groupby("participant_id")[nutrient_cols].mean()
nutr_mean.columns = [f"nutrient_daily_mean_{safe_name(c)}" for c in nutrient_cols]
density = daily_nutr[nutrient_cols[1:]].apply(lambda c: safe_divide(c, daily_nutr[ENERGY]) * 1000)
density = density.groupby(daily_nutr["participant_id"]).mean()
density.columns = [f"nutrient_per_1000kcal_{safe_name(c)}" for c in nutrient_cols[1:]]
nutrient_features = pd.concat([nutr_mean, density], axis=1).reset_index()

macro = daily_nutr[day_keys].copy()
for name, col, kcal in [("carb", "carbohydrate_g", 4), ("protein", "protein_g", 4), ("fat", "lipid_g", 9), ("alcohol", "alcohol_g", 7)]:
    macro[f"pct_{name}_calories"] = safe_divide(daily_nutr[col] * kcal, daily_nutr[ENERGY]) * 100
macro_cols = [c for c in macro.columns if c.startswith("pct_")]
macro_std = macro.groupby("participant_id")[macro_cols].agg(std0).add_suffix("_std")
macro_features = pd.concat([macro.groupby("participant_id")[macro_cols].mean(), macro_std], axis=1).reset_index()

In [ ]:
supp2_names = {canon(c): c for c in supp2.columns}
diet["_supp2_food"] = diet[FOOD].map(lambda f: f if f in supp2.columns else supp2_names.get(canon(f)))
matched = diet.dropna(subset=["_supp2_food"])
food_day = matched.pivot_table(index=day_keys, columns="_supp2_food", values=ENERGY, aggfunc="sum", fill_value=0)
day_index = food_day.index
day_energy = daily.set_index(day_keys)["daily_energy_kcal"].reindex(day_index).fillna(0)
day_category_energy = diet.pivot_table(index=day_keys, columns=CATEGORY, values=ENERGY, aggfunc="sum", fill_value=0).reindex(day_index).fillna(0)
day_category_weight = diet.pivot_table(index=day_keys, columns=CATEGORY, values=WEIGHT, aggfunc="sum", fill_value=0).reindex(day_index).fillna(0)
day_nutrients = daily_nutr.set_index(day_keys).reindex(day_index).fillna(0)


def supp2_foods(row, value=1):
    return [f for f, v in pd.to_numeric(supp2.loc[row], errors="coerce").fillna(0).items() if v == value and f in food_day]


def aligned_fraction(row):
    return safe_divide(food_day[supp2_foods(row)].sum(axis=1), day_energy).clip(0, 1)


def category_energy(categories):
    return cols_sum(day_category_energy, categories)


patterns = pd.DataFrame(index=day_index)
for name in ["Vegan", "Vegetarian", "Pescatarian"]:
    patterns[f"{name.lower()}_score_per_day"] = aligned_fraction(cfg.OTHER_PATTERN_CONFIG[name]["supp2_row"])

wfpb = cfg.OTHER_PATTERN_CONFIG["WFPB"]
shortfall = sum(((rule["threshold"] if rule["type"] == "absolute_kcal" else rule["threshold"] * day_energy)
                 - category_energy(rule["categories"])).clip(lower=0)
                for rule in wfpb["base_food_thresholds"].values())
patterns["wfpb_score_per_day"] = safe_divide(aligned_fraction(wfpb["supp2_row"]) * day_energy - shortfall, day_energy).clip(0, 1)

carnivore = cfg.OTHER_PATTERN_CONFIG["Carnivore"]
excess = sum((category_energy(rule["categories"]) - rule["threshold"] * day_energy).clip(lower=0)
             for rule in carnivore["moderation_food_thresholds"].values())
patterns["carnivore_score_per_day"] = safe_divide(aligned_fraction(carnivore["supp2_row"]) * day_energy - excess, day_energy).clip(0, 1)

patterns["NOVA_pct"] = 100 * safe_divide(food_day[supp2_foods("NOVA", 4)].sum(axis=1), day_energy).clip(0, 1)
carb_pct = safe_divide(day_nutrients["carbohydrate_g"] * 4, day_nutrients[ENERGY]) * 100
patterns["low_carb_diet"] = carb_pct.between(10, 26).astype(float)
patterns["extremely_low_carb"] = (carb_pct < 10).astype(float)

In [ ]:
eat = pd.DataFrame({
    comp: eat_component_score(cols_sum(day_category_weight if rule["unit"] == "g_per_day" else day_category_energy,
                                       rule["categories"]), rule)
    for comp, rule in cfg.EAT_LANCET_CONFIG.items()
}, index=day_index)

imedas = pd.DataFrame(index=day_index)
for comp, rule in cfg.IMEDAS_CONFIG.items():
    if rule["type"] == "difference_positive":
        diff = category_energy(rule["positive_categories"]) - category_energy(rule["negative_categories"])
        imedas[comp] = (diff > rule.get("threshold", 0)).astype(float)
    elif rule["type"] == "min_kcal_per_day":
        imedas[comp] = (category_energy(rule["categories"]) >= rule["threshold_kcal"]).astype(float)
    elif rule["type"] == "max_kcal_per_day":
        imedas[comp] = (category_energy(rule["categories"]) <= rule["threshold_kcal"]).astype(float)
    elif rule["type"] == "graded_max_kcal_per_day":
        x = category_energy(rule["categories"])
        score = pd.Series(0.0, index=day_index)
        for th in rule["thresholds"]:
            score[((x <= th["max_kcal"]) if th["max_kcal"] is not None else True) & (score == 0)] = th["points"]
        imedas[comp] = score

dash_values = {
    "total_fat_pct_energy": safe_divide(day_nutrients["lipid_g"] * 9, day_nutrients[ENERGY]) * 100,
    "protein_pct_energy": safe_divide(day_nutrients["protein_g"] * 4, day_nutrients[ENERGY]) * 100,
    "fiber_g": safe_divide(day_nutrients["dietary_fiber_g"], day_nutrients[ENERGY]) * 1000,
    "sodium_mg": safe_divide(day_nutrients["sodium_mg"], day_nutrients[ENERGY]) * 1000,
}
dash = pd.DataFrame(index=day_index)
for comp, rule in cfg.DASH_CONFIG.items():
    value = dash_values[rule["nutrient"]]
    better = value >= rule["target"] if rule["direction"] == "higher" else value <= rule["target"]
    middle = value >= rule["intermediate"] if rule["direction"] == "higher" else value <= rule["intermediate"]
    dash[comp] = np.where(better, 1.0, np.where(middle, 0.5, 0.0))

ahei = pd.DataFrame(index=day_index)
for comp in ["vegetables", "fruit", "sugar_sweetened_beverages_and_fruit_juice", "nuts_and_legumes", "red_processed_meat"]:
    rule = cfg.AHEI_2010_CONFIG[comp]
    score = linear_positive if rule["type"] == "linear_positive" else linear_negative
    ahei[comp] = score(category_energy(rule["categories"]) / rule["serving_kcal"],
                       rule["min_servings"], rule["max_servings"], rule["max_points"])
rule = cfg.AHEI_2010_CONFIG["whole_grains"]
ahei["whole_grains"] = linear_positive(cols_sum(day_category_weight, rule["categories"]), rule["min_value"],
                                       np.mean([rule["max_value_female"], rule["max_value_male"]]), rule["max_points"])
sodium = day_nutrients["sodium_mg"].groupby(level=0).mean()
sodium_decile = pd.qcut(sodium.rank(method="first"), 10, labels=list(range(10, 0, -1))).astype(float)
ahei["sodium"] = day_index.get_level_values(0).map(sodium_decile).to_numpy()

In [ ]:
category_share = category_energy_total.div(person_energy.reindex(category_energy_total.index), axis=0).fillna(0)
hpdi = pd.DataFrame(index=category_share.index)
for groups, direction in [("positive_groups", 1), ("less_healthy_plant_groups", -1), ("animal_based_groups", -1)]:
    for group, categories in cfg.HPDI_CONFIG[groups].items():
        q = quintile_positive(cols_sum(category_share, categories))
        hpdi[group] = q if direction == 1 else 6 - q
hpdi_score = hpdi.sum(axis=1).rename("hPDI_score_per_day")

daily_scores = pd.concat([
    patterns,
    eat.sum(axis=1).rename("eatLANCET_score_per_day"),
    imedas.sum(axis=1).rename("IMEDAS_score_per_day"),
    dash.sum(axis=1).rename("DASH_score"),
    ahei.sum(axis=1).rename("aHEI_score_per_day"),
], axis=1).reset_index()
score_cols = [c for c in daily_scores.columns if c not in day_keys]
daily_scores = daily_scores.merge(hpdi_score.rename_axis("participant_id").reset_index(), on="participant_id", how="left")
daily_scores.to_csv(OUT_DIR / "diet_daily_scores_baseline.csv", index=False)

score_groups = daily_scores.groupby("participant_id")[score_cols]
score_features = pd.concat([score_groups.mean(), score_groups.agg(std0).add_suffix("_std")], axis=1)
score_features = score_features.join(hpdi_score, how="outer").reset_index()

In [ ]:
diet_matrix = reduce(lambda a, b: a.merge(b, on="participant_id", how="outer"),
                     [behavior, food_features, category_features, nutrient_features, macro_features, score_features])
diet_matrix = diet_matrix.fillna(0)
constant = [c for c in diet_matrix.columns if c != "participant_id" and diet_matrix[c].nunique(dropna=False) <= 1]
diet_matrix = diet_matrix.drop(columns=constant)
diet_matrix.to_csv(OUT_DIR / "diet_feature_matrix_baseline_full.csv", index=False)
print("participants:", len(diet_matrix), "| dietary features:", diet_matrix.shape[1] - 1)